In [9]:
from dataset_loader import load_arxpr_data

from llama_index.core import Document
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes
from llama_index.core.node_parser import (SentenceSplitter, MarkdownNodeParser, MarkdownElementNodeParser)

from llama_index.core import VectorStoreIndex
from llama_index.core import Settings
from llama_index.core.schema import TextNode
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.ingestion import IngestionPipeline

from llama_index.core.extractors import (
    # SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
    # BaseExtractor,
)


import nest_asyncio
nest_asyncio.apply()



## Markdown Parser

In [2]:

def _pseudo_markdown_splitter(text: str, chunk_size=2000, chunk_overlap=125, markdown_headers=[]):
    """
    Borrowing the LangChain markdown splitter format to spot the # # ### and str; 
    keeps the header in medatdata dict if finds the header in markdown format, else nothing.
    Then recursively splits the text without breaking paragraphs
    
    Args:
        text: str
            The text from UnstructuredXMLLoader in line-separated format header, subheader, pargaraphs.
        ...
        headers_to_split_on: list
            A list of tuples of the form (header, metadata_key) where header is a str that
            will be used to split the text and metadata_key is the key in the metadata dict
            that will be used to store ONLY markdown-formatted header.

    Returns:
        splits: list
            A list of Langchain doc objects, each containing paragraphs and artifact sub-headers according to 
            chunk size; and 'metadata' key would contain real markdown headers IF any.
    
    """

    from langchain_text_splitters import MarkdownHeaderTextSplitter
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    # split by headers
    markdown_splitter = MarkdownHeaderTextSplitter(markdown_headers)
    md_header_splits = markdown_splitter.split_text(text)

    # chunk
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    splits = text_splitter.split_documents(md_header_splits)
    
    return splits


In [ ]:

def _match_sequence(seq1: str, seq2_list: list, threshold=0.7):
    """
    Gives a sequence similarity match between seq1 and seq2 from seq2_list.

    Args: 
        seq1: str
            The sequence to match
        seq_list: list
            A list of sequences to match against seq1
        threshold: float
            The minimum similarity ratio; <.8 for short sequences seems to work well.

    """

    from difflib import SequenceMatcher

    for seq2 in seq2_list:
        if SequenceMatcher(None, seq1, seq2).ratio() > threshold:
            return seq2
    return None
        


def split_non_md_headers(doc_lc, headers: list):
    """ A secondery cleaner to extract leftover non-markdown headers from the text. 
    
    Args:
        doc_lc: LangChain Document 
            A list of Langchain doc objects, each containing paragraphs and artifact sub-headers according to 
            chunk size; and 'metadata' key would contain real markdown headers IF any.
        headers: list    
            A list of possible headers (e.g. headers = ["introduction", "paragraph", "title_1", "title_2", "fig_caption"])
    Return:
        cleaned_docs: dict
            Contains core paragraphs under dict['text'] and associated headers under dict['headers']

    """


    cleaned_docs = {}
    cleaned_docs['text'] = ""
    cleaned_docs['headers'] = []

    text = doc_lc.page_content

    for line in text.split("\n"):
        match = _match_sequence(line.lower(), headers)
        if match:
            cleaned_docs['headers'].append(match)
        else:
            cleaned_docs['text'] += line


    return cleaned_docs

In [17]:
markdown_headers = [
    ("METHODS", "methods"),
    ("RESULTS", "result"),
    ("FIG", "figure"),
    ("INTRO", "introduction"),
    ("REF", "reference"),
    ("SUPPL", "supplement"),
    # ("DISCUSS", "discussion"),
    # ("CONCLUS", "conclusion"),
]

other_headers = ["introduction",
           "paragraph",
           "title_1",
           "title_2",
           "fig_caption",
           "abstract",
           "supplementary material",
           "materials and methods",
           "results and discussion",
           "footnote_title"]


t,l = load_arxpr_data(1)
text = t['25918225']


Settings.embed_model = OllamaEmbedding(model_name="llama3")
Settings.llm = Ollama(model="llama3") 


metadata_extractors = [
    # QuestionsAnsweredExtractor(questions=3),
    KeywordExtractor(keywords=5)
    ]

pipeline = IngestionPipeline(transformations=metadata_extractors)

nodes = []

splits = _pseudo_markdown_splitter(text, chunk_size=2000, chunk_overlap=125, markdown_headers=markdown_headers)

for doc in splits:
    split = split_non_md_headers(doc, other_headers)

    print(f'\nMD Headers: {doc.metadata} \nOther Headers: {split["headers"]} \nSize change: {len(doc.page_content)} ? {len(split["text"])}')
    print(f'{split["text"]}')

    nodes.append(TextNode(text=split['text']))

docs = pipeline.run(documents=nodes)

index = VectorStoreIndex(nodes=docs)

N datasets with exactly one label, for each field:
{'assay_by_molecule_14': 1,
 'assay_count_7': 1,
 'experimental_design_10': 0,
 'experimental_factors_20': 0,
 'hardware_4': 0,
 'name_19': 1,
 'no_of_samples_22': 0,
 'no_of_samples_23': 0,
 'organism_16': 1,
 'releasedate_12': 1,
 'sample_count_13': 1,
 'sex_2': 0,
 'study_type_18': 1,
 'technology_15': 1,
 'type_21': 0,
 'type_9': 1}
N datasets with at least one label, for each field
{'assay_by_molecule_14': 1,
 'assay_count_7': 1,
 'experimental_design_10': 0,
 'experimental_factors_20': 1,
 'hardware_4': 0,
 'name_19': 1,
 'no_of_samples_22': 0,
 'no_of_samples_23': 0,
 'organism_16': 1,
 'releasedate_12': 1,
 'sample_count_13': 1,
 'sex_2': 0,
 'study_type_18': 1,
 'technology_15': 1,
 'type_21': 1,
 'type_9': 1}

MD Headers: {} 
Other Headers: ['title_1', 'abstract', 'abstract', 'abstract', 'abstract'] 
Size change: 1061 ? 1004
BioC-APIcollection.keyCC BY-NC-SASplicing role of mitotic regulator kills tumor cellsThis article is d

100%|██████████| 36/36 [00:19<00:00,  1.88it/s]


In [18]:
from llama_index.core.response.notebook_utils import display_source_node

base_retriever = index.as_retriever(similarity_top_k=3)

query_str = "What is the use of RT-PCR kit?"
base_nodes = base_retriever.retrieve(query_str)


In [20]:
for node in base_nodes: 
    print('')
    # print(node.get_content(metadata_mode='all'))
    print(node.metadata)
    print(node.embedding)
    # display_source_node(node, source_length=500)


{'excerpt_keywords': 'Here are 5 unique keywords for this document:\n\nReal-time PCR, Quantitative RT-PCR, Drosophila, Gene expression, Alternative splicing'}
None

{'excerpt_keywords': "Here are five unique keywords that summarize the content of the document:\n\nRNase H1, BuGZ, DNA damage, p53 phosphorylation, cell stress\n\nLet me know if you'd like me to help with anything else!"}
None

{'excerpt_keywords': 'Here are five unique keywords that can be derived from the document:\n\nRNA-seq, Drosophila, Alternative splicing, Gene expression, Tumorogenesis'}
None


## Semantic Parsing

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=95, embed_model=embed_model
)
